In [8]:
import pandas as pd

# =====================================================
# LOAD DATASETS
# =====================================================

industry_df = pd.read_csv(
    "/Users/hamid/Desktop/SupplyChain_Project/data/cleaned/cleaned_industry_exposure.csv"
)

dim_industry = pd.read_csv(
    "/Users/hamid/Desktop/SupplyChain_Project/data/dimensions/dim_industry.csv"
)

dim_date = pd.read_csv(
    "/Users/hamid/Desktop/SupplyChain_Project/data/dimensions/dim_date.csv"
)

dim_era = pd.read_csv(
    "/Users/hamid/Desktop/SupplyChain_Project/data/curated/dim_disruption_era.csv"
)

# =====================================================
# CLEAN INDUSTRY NAMES
# =====================================================

industry_df["industry"] = (
    industry_df["industry"]
    .astype(str)
    .str.strip()
    .str.lower()
)

dim_industry["industry_name"] = (
    dim_industry["industry_name"]
    .astype(str)
    .str.strip()
    .str.lower()
)

# =====================================================
# JOIN industry_sk
# =====================================================

industry_df = industry_df.merge(
    dim_industry[
        ["industry_sk", "industry_name"]
    ],
    left_on="industry",
    right_on="industry_name",
    how="left"
)

# =====================================================
# DATE LOOKUP
# =====================================================

date_lookup = (
    dim_date
    .sort_values("date")
    .groupby("year")
    .first()
    .reset_index()[["year", "date_key"]]
)

industry_df = industry_df.merge(
    date_lookup,
    on="year",
    how="left"
)

# =====================================================
# ERA LOOKUP FUNCTION
# =====================================================

def assign_era_sk(year, era_df):

    matched_era = era_df[
        (era_df["start_year"] <= year) &
        (era_df["end_year"] >= year)
    ]

    if not matched_era.empty:
        return matched_era.iloc[0]["era_sk"]

    return None

# =====================================================
# ADD era_sk
# =====================================================

industry_df["era_sk"] = (
    industry_df["year"]
    .apply(lambda x: assign_era_sk(x, dim_era))
)

# =====================================================
# CREATE resilience_score
# =====================================================

industry_df["resilience_score"] = (
    10 - industry_df["overall_vulnerability"]
).round(2)

# =====================================================
# CREATE risk_level
# =====================================================

def classify_risk(vulnerability):

    if vulnerability < 3:
        return "Low Risk"

    elif vulnerability < 6:
        return "Medium Risk"

    elif vulnerability < 8:
        return "High Risk"

    return "Critical"

industry_df["risk_level"] = (
    industry_df["overall_vulnerability"]
    .apply(classify_risk)
)

# =====================================================
# CREATE FACT SURROGATE KEY
# =====================================================

industry_df.insert(
    0,
    "industry_fact_sk",
    range(1, len(industry_df) + 1)
)

# =====================================================
# ADD LOAD TIMESTAMP
# =====================================================

industry_df["load_timestamp"] = (
    pd.Timestamp.now()
    .strftime("%Y-%m-%d %H:%M:%S")
)

# =====================================================
# FINAL FACT TABLE
# =====================================================

fact_industry_exposure = industry_df[
    [
        "industry_fact_sk",
        "date_key",
        "industry_sk",
        "era_sk",
        "overall_vulnerability",
        "geopolitical_exposure",
        "resilience_score",
        "risk_level",
        "load_timestamp"
    ]
]

# =====================================================
# VALIDATION
# =====================================================

print("Missing industry_sk:",
      fact_industry_exposure["industry_sk"].isnull().sum())

print("Missing date_key:",
      fact_industry_exposure["date_key"].isnull().sum())

print("Missing era_sk:",
      fact_industry_exposure["era_sk"].isnull().sum())

# =====================================================
# EXPORT FACT TABLE
# =====================================================

fact_industry_exposure.to_csv(
    "/Users/hamid/Desktop/SupplyChain_Project/data/curated/fact_industry_exposure.csv",
    index=False
)

print("\nfact_industry_exposure.csv created successfully.\n")

print(fact_industry_exposure.head())

Missing industry_sk: 0
Missing date_key: 0
Missing era_sk: 0

fact_industry_exposure.csv created successfully.

   industry_fact_sk  date_key  industry_sk  era_sk  overall_vulnerability  \
0                 1  20000101            1       1                   7.72   
1                 2  20010101            1       1                   7.59   
2                 3  20020101            1       1                   8.13   
3                 4  20030101            1       1                   7.63   
4                 5  20040101            1       1                   7.84   

   geopolitical_exposure  resilience_score risk_level       load_timestamp  
0                   10.0              2.28  High Risk  2026-05-16 22:44:46  
1                   10.0              2.41  High Risk  2026-05-16 22:44:46  
2                   10.0              1.87   Critical  2026-05-16 22:44:46  
3                   10.0              2.37  High Risk  2026-05-16 22:44:46  
4                   10.0              2.